# Дослідження й рішення щодо очищення

Мета — зрозуміти якість початкових даних і пояснити правила підготовки. Моделі тут не навчаються. Виконання цього ноутбука не записує вихідних таблиць; для експорту є `02_prepare_data.ipynb`.

Кожен рядок — спір, за який продавець вже боровся і отримав фінальний результат. `won` — 1 для виграшу, 0 для програшу. Дані не описують усі можливі спори.

In [ ]:
from pathlib import Path
import sys

# Локально: запускаємо з папки data_preparation або її батьківської папки.
# У Colab задайте PROJECT_DIR — папку з preparation.py, а SOURCE — шлях до CSV.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "preparation.py").exists():
    PROJECT_DIR = PROJECT_DIR / "data_preparation"
SOURCE = PROJECT_DIR.parent / "dataset.csv"

if not (PROJECT_DIR / "preparation.py").exists():
    raise FileNotFoundError("Задайте PROJECT_DIR: папка з preparation.py")
if not SOURCE.exists():
    raise FileNotFoundError("Задайте SOURCE: шлях до початкового dataset.csv")
sys.path.insert(0, str(PROJECT_DIR.resolve()))
import pandas as pd
from IPython.display import display

from preparation import normalize_source, split_by_time, fit_scale_reference
raw = pd.read_csv(SOURCE)

## Початковий склад

Повний дублікат означає повтор усіх полів, включно з ID. Такі повтори видаляємо один раз. Повтор ID з різними значеннями не вирішуємо довільно: процес має зупинитися.

In [ ]:
display(pd.Series({
    "рядків": len(raw),
    "колонок": len(raw.columns),
    "повних_дублікатів": int(raw.duplicated().sum()),
    "повторних_ID": int(raw.dispute_id.duplicated().sum()),
}))
display(raw.won.value_counts(dropna=False).rename("спорів").to_frame())

## Категорії й типи

Прибираємо крайні пробіли; порожній рядок, `-` та `?` означають пропуск. Виправляємо відомі варіанти `pay pal`, `mastercrad`, `mc` і різний регістр. `reason_code` залишається текстом: значення на кшталт `10.4` не є числовою величиною.

Для PayPal пропущений бренд картки стає `not_card`, відповідно до словника набору. Для карткових оплат бренд не вгадуємо. Різні коди причин не об'єднуємо лише тому, що їхня категорія однакова.

In [ ]:
normalized, source_audit = normalize_source(raw)
display(pd.DataFrame({
    "тип": normalized.dtypes.astype(str),
    "пропусків": normalized.isna().sum(),
    "унікальних": normalized.nunique(),
}))
display(pd.crosstab(
    normalized.payment_method.fillna("MISSING"),
    normalized.card_brand.fillna("MISSING"),
))

In [ ]:
for column in ["psp", "reason_category", "reason_code", "currency"]:
    display(normalized[column].value_counts(dropna=False).rename("спорів").to_frame())

## Валюта `$`

`US$` трактуємо як USD, `€` — як EUR. Символ `$` неоднозначний. У цьому наборі його трактуємо як USD, оскільки в усіх 54 доступних парах суми збігаються. Для решти випадків це припущення, а не встановлений факт; прапорець `currency_dollar_assumed_usd` його зберігає.

`OTHER` означає рідкісну валюту, для якої оригінальну суму приховано. Відсутність `amount` тут очікувана. Відома додатна USD-сума може використовуватися без вгадування валюти.

In [ ]:
dollar = source_audit.currency_dollar_assumed_usd
pairs = normalized.loc[dollar, ["amount", "amount_usd"]].dropna()
display(pd.Series({
    "позначень_долара": int(dollar.sum()),
    "відомих_пар": len(pairs),
    "пар_з_однаковими_сумами": int(pairs.amount.sub(pairs.amount_usd).abs().le(.01).sum()),
    "відновлених_not_card": int(source_audit.card_brand_restored.sum()),
    "конфліктів_оплати_та_бренду": int(source_audit.payment_brand_conflict.sum()),
}))

## Суми до виправлення

Нуль, від'ємна сума та повторювані значення `9 999 999` / `999 999` не використовуються як надійні суми спору. Вони стають пропусками. Це обережне припущення: знак міг мати бухгалтерський зміст, а походження заглушок невідоме. Модуль від'ємного числа не беремо.

Рядки не видаляємо. Початкові значення залишаються в аудиті. Квантили `amount` змішують валюти й не характеризують єдину грошову шкалу.

In [ ]:
display(normalized[["amount", "amount_usd"]].describe(percentiles=[.25, .5, .75, .95, .99]).T)
source_flags = [column for column in source_audit if column.startswith("source__amount")]
display(source_audit[source_flags].sum().rename("спорів").to_frame())

## Часові частини

День тижня має бути цілим 0–6. Тиждень — невід'ємним цілим; `999` і `2026` у цьому наборі трактуємо як підозрілі службові значення. Некоректний або невідомий тиждень означає окрему частину `unknown_time`.

Train — ранній період, на якому визначаємо статистичні правила. Validation — наступний період для подальшого порівняння рішень команди. Test — найпізніший період для підсумкової перевірки. Навчання моделі для цього поділу не потрібне.

Початкова чернетка вже показувала загальну статистику й результати за тижнями до поділу. Test не можна називати повністю недоторканим. Надалі не підбираємо за ним правила.

In [ ]:
parts = split_by_time(normalized)
display(pd.DataFrame([
    {"частина": name, "рядків": len(part),
     "перший_тиждень": part.dispute_week.min(), "останній_тиждень": part.dispute_week.max()}
    for name, part in parts.items()
]))

## Гіпотеза помилки масштабу

Опорні співвідношення `amount_usd / amount` визначаємо лише на train. Беремо додатні скінченні пари без заглушок, обидві суми від 10, мінімум 30 пар на валюту. Межі — 5-й і 95-й процентилі з запасом 5%.

USD-суму ділимо на 1000 лише коли початкове відношення перевищує типову медіану щонайменше у 100 разів, а виправлене потрапляє в межі. Для USD вимагаємо збіг двох сум із допуском 0.01.

Це вузьке правило для конкретної аномалії. Воно не доводить, що помилка саме в USD-колонці, не виправляє всі можливі масштаби й не є довідником валютних курсів. PLN має замало пар для автоматичного правила; ILS із 32 парами — слабша опора, ніж USD.

In [ ]:
reference = fit_scale_reference(parts["train"])
display(reference[["pairs", "median", "lower", "upper"]])

## Відновлення та невирішені суми

Коли валюта USD, одну відсутню суму відновлюємо з іншої, якщо джерело додатне і не перевищує 10 000. Для інших валют таке відновлення не виконуємо.

Після корекції й відновлення суму понад 10 000 залишаємо лише в аудиті, а в основній таблиці робимо пропуском. Для `amount` поріг задано в оригінальній валюті. При конфлікті двох USD-сум не обираємо довільно правильну: обидві стають пропусками. Поріг є робочим правилом перевірки, не доведеним максимумом платежу.

У поточному train це прибирає з основної USD-колонки два невирішені значення — 46 000 і 11 000. У попередньому архіві вони залишалися числами, хоча були виключені зі Score.

Окремі аудиторські колонки зберігають походження суми: спостережена, виправлена за масштабом, відновлена, недоступна або потребує перевірки.

## Що залишається наступному учаснику

Пропуски збережені. Заповнення медіаною, кодування категорій, масштабування ознак і вибір моделі не належать до цього етапу. Вони мають налаштовуватися лише на train. Початкові суми, ID, ціль і прапорці оцінювання не слід автоматично включати у вхід моделі.

Історична вибірка охоплює тільки спори, за які вже боролися й отримали результат. Відсутня дата отримання результату: не доведено, що всі мітки ранніх спорів були відомі на початку наступного періоду. Це обмеження даних, яке очищення не усуває.

Експорт і технічні перевірки — у наступному ноутбуці.